<a href="https://colab.research.google.com/github/aldo02032004/sentiment-volume-forecast/blob/main/Linear_regression_forecast.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### Panduan pengaturan ideal

| Panjang data | Prediksi 1 – 3 hari | Prediksi 4 – 7 hari | Catatan |
|---|---|---|---|
| < 21 hari | – | – | data terlalu pendek, belum bisa diuji |
| 21 – 45 hari | belajar **7** | belajar **7** | skor masih kasar (dinilai < 10 kali uji) |
| 46 – 89 hari | belajar **7 – 14** | belajar **14** | |
| ≥ 90 hari | belajar **7 – 42** | belajar **14 – 42** | volume stabil → 28–42; volume lagi naik/turun → 7–14 |

**Tanda "Hari dipelajari" perlu dikecilkan:** Coverage jauh di bawah 80%, atau "Asli > tebakan" jauh dari 50% (di bawah 35% atau di atas 65%). Artinya volume lagi berubah dan model telat ngikutin.

**Jumlah uji** = panjang data − hari dipelajari − hari diprediksi + 1 *(minimal 14, maksimal 60 hari terakhir)*.
**2/3 uji pertama** dipakai buat milih metode & bikin area bayangan, **1/3 terakhir** buat nilai skor.

# 1. Install

In [ ]:
!pip install -q sktime "great[viz] @ git+https://github.com/azmkto/GREAT-Tools.git"

# 2. Library


In [ ]:
import io, re, time, zipfile          # io: baca file dari memori ; re: ambil ID dari link drive
                                      # time: jeda sebelum unduh ulang ; zipfile: buka file zip berisi Excel

import numpy as np                        # hitungan angka
import pandas as pd                       # tabel data
import requests                           # unduh file dari Google Drive
import ipywidgets as w                    # slider, dropdown, tombol
import matplotlib.pyplot as plt           # gambar grafik
from matplotlib.lines import Line2D       # contoh garis putus-putus di keterangan grafik
from matplotlib.patches import Patch      # contoh area bayangan di keterangan grafik
from IPython.display import display       # tampilin tabel & interaktif
from sklearn.linear_model import Ridge   # scikit-learn: Ridge = regresi linear dengan penahan
from sklearn.metrics import mean_absolute_error, root_mean_squared_error   # MAE, RMSE (WAPE = MAE / rata-rata asli)
from sktime.split import ExpandingWindowSplitter                           # bagi titik uji per waktu (rolling origin)
from sktime.performance_metrics.forecasting import mean_absolute_scaled_error   # MASE

from great import sent_class, sent_colors, validate_export   # library GREAT: urutan & warna sentimen, format file export
from great.viz import prep                                   # ngerapiin data export (tanggal, sentimen, dll.)
from great.viz.overview import _panel_sentiment_trend        # grafik tren sentimen yang sama dengan monthly report
from great.viz.style import apply_style                      # grafik style

print("Library siap")

# 3. Pengaturan


In [ ]:
SOURCES = [   # link Google Drive (zip, excel,sheets). Default: data contoh, ganti dengan link kamu
    "https://raw.githubusercontent.com/aldo02032004/sentiment-volume-forecast/main/examples/sample_export.xlsx",
]
OBJECT_NAME = "Someone"   # nama objek penelitian

METHODS = ["Hari terakhir", "Rata-rata", "Minggu lalu", "Regresi pintar", "Gabungan"]
TEST_DAYS = 60        # uji ketepatan ke 60 hari terakhir
MIN_TESTS = 14        # minimal jumlah uji supaya skor bisa dipercaya
BAND = 0.80           # lebar area bayangan: 80% angka asli masuk area ini
RIDGE_ALPHA = 10      # seberapa kuat Regresi pintar menahan efek per hari

# 4. Load data

In [ ]:
def download(link, label, tries=3):
    match = re.search(r"/d/([\w-]+)", link) or re.search(r"[?&]id=([\w-]+)", link)
    urls = [link]   # link biasa (mis. data contoh di GitHub) langsung diunduh
    if match:       # link Google Drive / Sheets
        urls = [f"https://drive.usercontent.google.com/download?id={match.group(1)}&export=download&confirm=t",  # file Excel / zip
                f"https://docs.google.com/spreadsheets/d/{match.group(1)}/export?format=xlsx"]                    # Google Sheets
    for attempt in range(1, tries + 1):
        for url in urls:
            try:
                resp = requests.get(url, timeout=300)
            except requests.RequestException:
                continue
            if resp.ok and resp.content[:2] in (b"PK", b"\xd0\xcf"):   # cek beneran file Excel / zip, bukan halaman error
                return resp.content
        time.sleep(3 * attempt)   # jeda sebelum coba lagi
    raise RuntimeError(f"{label} gagal diunduh. Cek akses 'Anyone with the link' & koneksi internet.")

def load_link(link, label):
    files = [download(link, label)]
    if files[0][:2] == b"PK":   # .xlsx atau .zip (dua-duanya format zip)
        with zipfile.ZipFile(io.BytesIO(files[0])) as z:
            names = z.namelist()
            if "xl/workbook.xml" not in names:   # zip isi beberapa file Excel, skip file sampah Mac & file sementara Excel
                files = [z.read(n) for n in names if n.lower().endswith((".xlsx", ".xls"))
                         and not n.startswith("__MACOSX/") and "/~$" not in "/" + n]
                print(f"{label}: zip isi {len(files)} file Excel")
    return [pd.read_excel(io.BytesIO(f), header=1) for f in files]


df = pd.concat([validate_export(t) for i, s in enumerate(SOURCES, 1) for t in load_link(s, f"file_{i}")],
               ignore_index=True)
df["Date"] = pd.to_datetime(df["Date"], format="ISO8601")
before = len(df)
df = df.drop_duplicates(subset=[c for c in df.columns if c != "No"]).reset_index(drop=True)   # buang postingan double dari file yang tanggalnya tumpang tindih
df1, start_date, end_date = prep.prepare_data(df)
print(f"data yang digunakan: {len(df1):,} baris ({before - len(df):,} terindikasi redundan di cut) | {start_date} - {end_date}")

# 5. Hitung per hari


In [ ]:
group = df1.groupby([pd.Grouper(key="Date", freq="D"), "Sentiment"])
daily = {
    "Postingan": group.size(),               # jumlah postingan per hari
    "Akun": group["Author"].nunique(),       # jumlah akun berbeda per hari
}
daily = {k: v.unstack(fill_value=0).reindex(columns=sent_class, fill_value=0).asfreq("D", fill_value=0).astype(float)
         for k, v in daily.items()}

n_days = len(daily["Postingan"])
if n_days < 7 + MIN_TESTS:                   # minimal 7 hari dipelajari + MIN_TESTS kali uji
    raise ValueError(f"Data cuma {n_days} hari, minimal {7 + MIN_TESTS} hari")
print(f"Total data: {n_days} hari")
empty = daily["Postingan"].index[daily["Postingan"].sum(axis=1) == 0]   # hari tanpa satu pun postingan = kemungkinan file export bolong, bukan beneran 0
if len(empty):
    print(f"Cek dulu, {len(empty)} hari kosong total: " + ", ".join(f"{d:%d %b %Y}" for d in empty[:10]))
for k, v in daily.items():
    print(f"\n{k} 10 hari terakhir:")
    display(v.tail(10).astype(int))

# 6. Metode prediksi


In [ ]:
# fitur kalender per hari: [Senin..Minggu] isinya 1/0 (akhir pekan sudah terwakili Sabtu + Minggu)
def calendar(dates):
    dow = dates.dayofweek.to_numpy()
    return (dow[:, None] == np.arange(7)).astype(float)


# prediksi `horizon` hari ke depan, belajar dari `lookback` hari terakhir
def forecast(y, method, lookback, horizon):
    hist = y.iloc[-lookback:]
    future = pd.date_range(y.index[-1] + pd.Timedelta(days=1), periods=horizon, freq="D")

    if method == "Hari terakhir":         # besok = sama dengan hari terakhir
        pred = np.tile(hist.iloc[-1], (horizon, 1))
    elif method == "Rata-rata":           # besok = rata-rata hari yang dipelajari
        pred = np.tile(hist.mean(), (horizon, 1))
    elif method == "Minggu lalu":         # besok = hari yang sama minggu lalu
        pred = y.iloc[-7:].to_numpy()[np.arange(horizon) % 7]
    elif method == "Regresi pintar":
        # log(1 + hari ini) ditebak dari log(1 + kemarin) + efek hari; tebakan besok dipakai buat lusa, dst.
        z, cal, pred = np.log1p(hist.to_numpy()), calendar(hist.index), []
        for j in range(z.shape[1]):   # satu model per sentimen
            model = Ridge(alpha=RIDGE_ALPHA).fit(np.column_stack([z[:-1, j], cal[1:]]), z[1:, j])
            last, out = z[-1, j], []
            for c in calendar(future):
                last = model.intercept_ + model.coef_ @ np.r_[last, c]
                out.append(last)
            pred.append(np.expm1(out))
        pred = np.column_stack(pred)
    elif method == "Gabungan":            # rata-rata Regresi pintar & Hari terakhir: akurat tapi tetap cepat ngikutin perubahan
        return (forecast(y, "Regresi pintar", lookback, horizon) + forecast(y, "Hari terakhir", lookback, horizon)) / 2
    else:
        raise ValueError(f"Metode tidak dikenal: {method}")
    return pd.DataFrame(np.clip(pred, 0, None), index=future, columns=y.columns)   # jumlah nggak mungkin minus


# uji ke data lama (rolling origin): tiap uji cuma lihat data sebelum hari yang ditebak
def backtest(y, methods, lookback, horizon, test_days):
    start = max(lookback, 7, len(y) - horizon + 1 - test_days)
    cv = ExpandingWindowSplitter(initial_window=start, step_length=1, fh=list(range(1, horizon + 1)))
    ends = [len(train) for train, _ in cv.split(y)]   # panjang data latih di tiap uji
    actual = np.array([y.iloc[end:end + horizon].to_numpy() for end in ends])   # bentuk: (uji, hari ke depan, sentimen)
    errors = {m: actual - np.array([forecast(y.iloc[:end], m, lookback, horizon).to_numpy() for end in ends])
              for m in methods}                                                  # angka asli - prediksi
    return errors, actual


# ketepatan = 1 - WAPE ; WAPE = MAE / rata-rata angka asli
def score(err, act):
    return 1 - mean_absolute_error(act.ravel(), (act - err).ravel()) / act.mean()


print("Metode siap:", ", ".join(METHODS))

# 7. Visual

| Kolom skor | Arti | Bagus kalau |
|---|---|---|
| Ketepatan | 1 − WAPE | makin tinggi |
| MASE | meleset dibanding tebakan "sama kayak kemarin" | < 1 |
| MAE / RMSE | rata-rata meleset per hari per sentimen (RMSE lebih menghukum lonjakan) | makin kecil |
| Asli > tebakan | seberapa sering angka asli di atas tebakan | ±50% |
| Coverage | seberapa sering angka asli masuk area bayangan | ±80% |
| Lebar bayangan | rata-rata lebar area bayangan | makin sempit, asal Coverage tetap ±80% |

In [ ]:
# pengaturan tampilan grafik
FIGSIZE = (30, 8)          # (lebar, tinggi) grafik
DPI = 100                  # ketajaman grafik: makin besar makin tajam, tapi slider makin lambat
COLOR_LEARNED = "orange"   # warna kotak hari yang dipelajari
COLOR_LEGEND = "grey"      # warna contoh garis & bayangan di keterangan grafik
AUTO = "Otomatis (paling akurat)"


# rentang aman slider, ngikutin panjang data
def max_horizon(n):
    # prediksi paling jauh: 1/4 panjang data, maksimal 14 hari, dan masih nyisain MIN_TESTS kali uji
    return max(1, min(14, n // 4, n - 6 - MIN_TESTS))


def lookback_range(n, horizon):
    low = max(2, min(7, n - horizon))   # minimal 7 hari (1 minggu penuh)
    return low, max(low, n - horizon + 1 - MIN_TESTS)   # maksimal: masih nyisain MIN_TESTS kali uji


# hasil uji disimpan, jadi kalau slider digeser balik ke posisi lama nggak ngitung ulang
_cache = {}


# uji semua metode, pilih yang terbaik, bikin area bayangan tiap metode
def run(basis, horizon, lookback, method):
    key = (basis, horizon, lookback)
    if key not in _cache:
        _cache[key] = backtest(daily[basis], METHODS, lookback, horizon, TEST_DAYS)
    errors, actual = _cache[key]
    pick = len(actual) * 2 // 3   # 2/3 uji pertama buat milih metode & bikin bayangan, 1/3 terakhir buat nilai (biar skornya jujur)

    choose = {m: score(e[:pick], actual[:pick]) for m, e in errors.items()}
    best = method if method in choose else max(choose, key=choose.get)   # Otomatis -> metode paling akurat
    # area bayangan = sebaran meleset di uji data lama (per hari ke depan), nanti ditempel ke prediksi
    bands = {m: np.quantile(e[:pick], [(1 - BAND) / 2, (1 + BAND) / 2], axis=0) for m, e in errors.items()}
    return best, errors, actual, pick, bands


# skor tiap metode, dinilai di 1/3 uji terakhir yang nggak dipakai buat milih metode & bikin bayangan
def score_table(y, horizon, errors, actual, pick, bands):
    test, act = {m: e[pick:] for m, e in errors.items()}, actual[pick:]
    train = y.iloc[:len(y) - horizon + 1 - len(actual)]   # data sebelum uji pertama (buat skala MASE)
    table = lambda x: pd.DataFrame(x.reshape(-1, x.shape[2]))   # (uji, hari ke depan, sentimen) -> tabel: kolom = sentimen
    return pd.DataFrame({
        "Ketepatan": {m: score(e, act) for m, e in test.items()},                               # 100% - rata-rata meleset (WAPE)
        "MASE": {m: mean_absolute_scaled_error(table(act), table(act - e), y_train=train.reset_index(drop=True), sp=1)
                 for m, e in test.items()},                                                     # < 1 = lebih bagus dari tebakan "sama kayak kemarin"
        "MAE": {m: mean_absolute_error(act.ravel(), (act - e).ravel()) for m, e in test.items()},       # rata-rata meleset (jumlah per hari per sentimen)
        "RMSE": {m: root_mean_squared_error(act.ravel(), (act - e).ravel()) for m, e in test.items()},  # meleset besar (lonjakan) dihukum lebih berat
        "Asli > tebakan": {m: (e > 0).mean() for m, e in test.items()},                        # ±50% = tebakan tipe median
        "Coverage": {m: ((e >= bands[m][0]) & (e <= bands[m][1])).mean() for m, e in test.items()},  # idealnya ±80%
        "Lebar bayangan": {m: (bands[m][1] - bands[m][0]).mean() for m in test},                 # makin sempit makin berguna, asal Coverage tetap ±80%
    }).sort_values("Ketepatan", ascending=False)


# pilihan interaktif, batas slider otomatis di rentang aman
apply_style(dpi=DPI)
style = {"description_width": "140px"}
basis_w = w.ToggleButtons(options=list(daily), description="Yang dihitung", style=style)
method_w = w.Dropdown(options=[AUTO] + METHODS, value=AUTO, description="Metode", style=style)
horizon_w = w.IntSlider(value=min(3, max_horizon(n_days)), min=1, max=max_horizon(n_days),
                        description="Hari diprediksi", style=style)
low_lb, high_lb = lookback_range(n_days, horizon_w.value)
lookback_w = w.IntSlider(value=min(7, high_lb), min=low_lb, max=high_lb, description="Hari dipelajari", style=style)


# kalau "hari diprediksi" digeser, rentang "hari dipelajari" ikut nyesuaiin
def update_lookback(change):
    low_lb, high_lb = lookback_range(n_days, horizon_w.value)
    if low_lb > lookback_w.max:
        lookback_w.max, lookback_w.min = high_lb, low_lb
    else:
        lookback_w.min, lookback_w.max = low_lb, high_lb


horizon_w.observe(update_lookback, names="value")


def draw(basis, horizon, lookback, method):
    y = daily[basis]
    best, errors, actual, pick, bands = run(basis, horizon, lookback, method)
    pred = forecast(y, best, lookback, horizon)
    low, high = (pred + bands[best][0]).clip(lower=0), (pred + bands[best][1]).clip(lower=0)

    fig, ax = plt.subplots(figsize=FIGSIZE)
    _panel_sentiment_trend(ax, y, sent_class, sent_colors, OBJECT_NAME, start_date, end_date, interval=1)
    ax.set_title(f"Sentiment of {OBJECT_NAME}\n{y.index[0]:%d %b %Y} - {pred.index[-1]:%d %b %Y}",
                 fontsize=20, fontweight="bold", pad=20)

    line = pd.concat([y.iloc[[-1]], pred])   # mulai dari titik asli terakhir biar garisnya nyambung
    for s in sent_class:
        ax.plot(line.index, line[s], color=sent_colors[s], linewidth=3, linestyle="--", marker="o", ms=3)
        ax.fill_between(pred.index, low[s], high[s], color=sent_colors[s], alpha=0.12, linewidth=0)

    # angka di ujung garis, kalau tumpuk yang di atas digeser naik
    y_min, y_max = ax.get_ylim()
    gap, prev = 0.05 * (y_max - y_min), -np.inf
    for s in sorted(sent_class, key=lambda s: pred[s].iloc[-1]):
        prev = max(pred[s].iloc[-1], prev + gap)
        ax.annotate(f"{pred[s].iloc[-1]:,.0f} ({low[s].iloc[-1]:,.0f}–{high[s].iloc[-1]:,.0f})",
                    (pred.index[-1], prev), xytext=(8, 0), textcoords="offset points",
                    va="center", fontsize=12, color=sent_colors[s], fontweight="bold")
    ax.axvspan(y.index[-lookback], y.index[-1], color=COLOR_LEARNED, alpha=0.12)

    # keterangan grafik: warna sentimen + arti garis putus-putus, bayangan, kotak oranye
    handles, labels = ax.get_legend_handles_labels()
    handles += [Line2D([], [], color=COLOR_LEGEND, linewidth=3, linestyle="--"),
                Patch(color=COLOR_LEGEND, alpha=0.25), Patch(color=COLOR_LEARNED, alpha=0.25)]
    labels += ["Prediksi", f"Rentang wajar ({BAND:.0%} yakin)", f"Hari yang dipelajari ({lookback} hari)"]
    ax.legend(handles, labels, fontsize=12, loc="upper center", bbox_to_anchor=(0.5, 1.05), frameon=False, ncol=len(labels))

    dates = y.index.append(pred.index)
    ax.set_xticks(dates[::max(1, len(dates) // 15)])   # maksimal ~15 tanggal di sumbu bawah
    ax.set_xlabel("Tanggal", fontsize=18, fontweight="semibold")
    ax.set_ylabel(f"{basis} per hari", fontsize=18, fontweight="semibold")
    plt.tight_layout()
    plt.show()

    scores = score_table(y, horizon, errors, actual, pick, bands)
    print(f"Metode: {best}" + (" (otomatis)" if method == AUTO else "") +
          f" | ketepatan {scores.loc[best, 'Ketepatan']:.0%} | dipilih dari {pick} uji, dinilai di {len(actual) - pick} uji terakhir")
    display(scores.style.format({"Ketepatan": "{:.0%}", "MASE": "{:.2f}", "MAE": "{:,.0f}", "RMSE": "{:,.0f}",
                                 "Asli > tebakan": "{:.0%}", "Coverage": "{:.0%}", "Lebar bayangan": "{:,.0f}"})
                        .set_properties(**{"text-align": "center"})
                        .set_table_styles([{"selector": "th", "props": [("text-align", "center")]}]))   # isi & judul kolom di tengah
    if len(actual) < MIN_TESTS:
        print(f"Data masih pendek (uji < {MIN_TESTS} kali), skor ketepatan masih kasar.")


out = w.interactive_output(draw, {"basis": basis_w, "horizon": horizon_w, "lookback": lookback_w, "method": method_w})
display(w.VBox([w.HBox([basis_w, method_w]), w.HBox([horizon_w, lookback_w]), out]))